# Inspect FSD Cube muon selection

This notebook calls the production package. It does not produce or modify simulation files.
Run from `ND_Production/run-muon-skim`; NumPy, h5py, PyYAML, and Matplotlib are needed.
The truth length cut is a preselection; the reconstructed 90-hit and shape cuts still apply.


In [ ]:
from pathlib import Path
import json
import sys
import h5py
import numpy as np

stage = Path.cwd()
if not (stage / 'muon_skim').is_dir():
    stage = stage / 'run-muon-skim'
if not (stage / 'muon_skim').is_dir():
    raise RuntimeError('Start this notebook in ND_Production or run-muon-skim')
sys.path.insert(0, str(stage.resolve()))
from muon_skim import load_config, cube_geometry, translations
from muon_skim.geometry import fingerprint
from muon_skim.io import read_tables, events
from muon_skim.selection import qualifying_translations, choose_translation


In [ ]:
# Edit paths here. None leaves that optional inspection disabled.
larnd_repo = stage.resolve().parents[1] / 'larnd-sim'
source_path = None  # e.g. Path('/path/to/input.EDEPSIM.hdf5')
source_event_id = None  # None means the first event containing segments
skim_directory = None  # e.g. Path('/path/to/completed-skim')
config = load_config(stage / 'configs/fsd_cube.json')
# To preview an override: config['min_length_cm'] = 29.0


In [ ]:
bounds, geometry_sources = cube_geometry(larnd_repo, config['detector_config'])
shifts = translations(config)
print('Target bounds [cm], rows x/y/z:', bounds, sep='\n')
print('Strict minimum length [cm]:', config['min_length_cm'])
print('Candidate translations:', len(shifts))


In [ ]:
event = choice = None
if source_path is not None:
    info = fingerprint(source_path)
    with h5py.File(source_path, 'r') as source:
        for event_id, rows in events(read_tables(source)):
            if source_event_id is None or event_id == source_event_id:
                event = rows
                candidates = qualifying_translations(rows['segments'], shifts, bounds, config['min_length_cm'])
                choice = choose_translation(candidates, config['seed'], info['sha256'], event_id)
                print('Source event:', event_id)
                print('Qualifying placements:', len(candidates))
                print(json.dumps(choice, indent=2))
                break
        else:
            raise ValueError('Requested event not found')
else:
    print('Set source_path to inspect a source event.')


In [ ]:
if choice is not None:
    import itertools
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d.art3d import Line3DCollection

    rows = event['segments']
    # Plot all muon segments, preserving the event; selection/writing is in the package.
    rows = rows[np.abs(rows['pdg_id'].astype(np.int64)) == 13]
    starts = np.column_stack([rows[f'{a}_start'] for a in 'xyz']) + choice['shift_cm']
    ends = np.column_stack([rows[f'{a}_end'] for a in 'xyz']) + choice['shift_cm']
    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(projection='3d')
    # Display y vertically: plotted axes are x, z, y, explicitly labeled below.
    ax.add_collection3d(Line3DCollection(np.stack([starts[:, [0, 2, 1]], ends[:, [0, 2, 1]]], axis=1), colors='tab:blue', linewidths=1))
    corners = np.array(list(itertools.product(*bounds)))
    edges = [np.stack([a[[0, 2, 1]], b[[0, 2, 1]]]) for a, b in itertools.combinations(corners, 2) if np.count_nonzero(a != b) == 1]
    ax.add_collection3d(Line3DCollection(edges, colors='black'))
    ax.set(xlim=bounds[0], ylim=bounds[2], zlim=bounds[1], xlabel='x drift [cm]', ylabel='z transverse [cm]', zlabel='y vertical [cm]', title='Selected translation: muon deposits and Cube bounds')
    ax.set_box_aspect(np.diff(bounds, axis=1).ravel()[[0, 2, 1]])
    plt.show()


In [ ]:
if skim_directory is not None:
    manifest = json.loads((Path(skim_directory) / 'manifest.json').read_text())
    print({k: manifest[k] for k in ('status', 'scanned_events', 'selected_events')})
    for output in manifest['outputs'][:3]:
        with h5py.File(output['path']) as skim:
            print(output['path'], 'events:', output['events'])
            record = json.loads(skim['provenance/events'][0])
            print({k: v for k, v in record.items() if k != 'id_maps'})
            print('ID map columns:', skim['provenance'].attrs['id_map_columns_json'])


The original shower notebook is retained separately as a reference. Production runs use
`python -m muon_skim`, never this notebook. Geometry comes from the specified YAML checkout;
this plot is an active-volume schematic, not a new Geant4 transport calculation.
